In [1]:
!nvidia-smi

Mon Aug 31 08:40:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import subprocess, sys

TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
)

installing: transformers==4.46.* accelerate==1.1.*


In [3]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA version:", torch.version.cuda)

Mon Aug 31 08:41:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)

tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda"
)

print("Model loaded!")
print("GPU:", torch.cuda.get_device_name(0))
print("dtype:", next(model.parameters()).dtype)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!
GPU: Tesla T4
dtype: torch.float16


In [5]:
import time, threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")

    streamer = TextIteratorStreamer(
        tok,
        skip_prompt=True,
        skip_special_tokens=True
    )

    kwargs = dict(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        streamer=streamer
    )

    th = threading.Thread(target=model.generate, kwargs=kwargs)

    t0 = time.time()
    th.start()

    stamps = []

    for _ in streamer:
        stamps.append(time.time())

    th.join()

    ttft = stamps[0] - t0

    if len(stamps) > 1:
        gaps = [b - a for a, b in zip(stamps, stamps[1:])]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0

    total = stamps[-1] - t0

    return {
        "ttft_s": round(ttft, 4),
        "tpot_s": round(tpot, 4),
        "total_s": round(total, 4),
        "n_tokens": len(stamps)
    }

In [6]:
measure_stream(prompt_of_len(128), new_tokens=8)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'ttft_s': 3.0569, 'tpot_s': 0.1178, 'total_s': 3.9989, 'n_tokens': 9}

In [7]:
ttft_by_len = {}

for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(n, r)

128 {'ttft_s': 0.0608, 'tpot_s': 0.0345, 'total_s': 4.4762, 'n_tokens': 129}
512 {'ttft_s': 0.067, 'tpot_s': 0.0324, 'total_s': 4.2148, 'n_tokens': 129}
2048 {'ttft_s': 0.3316, 'tpot_s': 0.0378, 'total_s': 5.1667, 'n_tokens': 129}


In [8]:
import gc

def kv_formula_kb_per_token(
    layers=28,
    kv_heads=2,
    head_dim=128,
    dbytes=2
):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024


def cache_bytes(pkv):
    if hasattr(pkv, "key_cache"):
        tensors = list(pkv.key_cache) + list(pkv.value_cache)
    else:
        tensors = [t for layer in pkv for t in layer]

    return sum(
        t.numel() * t.element_size()
        for t in tensors
    )


def measure_kv(context: int, new_tokens: int = 256):
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()

    enc = tok(
        prompt_of_len(context),
        return_tensors="pt"
    ).to("cuda")

    before = torch.cuda.memory_allocated()

    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        use_cache=True,
        return_dict_in_generate=True
    )

    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated()

    total_tokens = out.sequences.shape[1]

    return {
        "context": context,
        "total_tokens": int(total_tokens),

        "peak_kb_per_token": round(
            (peak - before) / total_tokens / 1024, 1
        ),

        "kv_kb_per_token": round(
            cache_bytes(out.past_key_values)
            / total_tokens / 1024, 1
        ),
    }

In [9]:
formula = kv_formula_kb_per_token()

print("formula KB/token:", formula)

kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]

for r in kv_rows:
    print(r, " vs formula", formula, "KB/token")

formula KB/token: 28.0


From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 63.4, 'kv_kb_per_token': 28.0}  vs formula 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 84.0, 'kv_kb_per_token': 28.0}  vs formula 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 87.6, 'kv_kb_per_token': 28.0}  vs formula 28.0 KB/token


In [10]:
import json

with open("kv_check.json", "w") as f:
    json.dump({
        "formula_kb_per_token": formula,
        "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
        "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"]
    }, f)

print("kv_check.json created!")

kv_check.json created!


In [11]:
QUEUE = [32, 32, 32, 256] * 6

def static_queue(
    batch: int,
    prompt: str = "Explain what an inference server does."
):
    t0 = time.time()
    useful = 0
    slots = 0

    for i in range(0, len(QUEUE), batch):
        chunk = QUEUE[i:i + batch]

        # Batch runs until the longest request finishes
        n = max(chunk)

        enc = tok(
            [prompt] * len(chunk),
            return_tensors="pt",
            padding=True
        ).to("cuda")

        model.generate(
            **enc,
            max_new_tokens=n,
            do_sample=False
        )

        useful += sum(chunk)
        slots += n * len(chunk)

    dt = time.time() - t0

    return {
        "batch": batch,
        "wall_s": round(dt, 2),
        "tokens_per_s": round(useful / dt, 1),
        "slot_efficiency": round(useful / slots, 3)
    }

In [12]:
batch_rows = {}

for n in [1, 4, 8]:
    r = static_queue(n)
    batch_rows[str(n)] = r["tokens_per_s"]
    print(r)

{'batch': 1, 'wall_s': 60.92, 'tokens_per_s': 34.7, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 40.12, 'tokens_per_s': 52.6, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 20.13, 'tokens_per_s': 104.9, 'slot_efficiency': 0.344}


In [13]:
import json

baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {k: v for k, v in batch_rows.items()},
}

with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)

print(json.dumps(baselines, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.0608,
    "512": 0.067,
    "2048": 0.3316
  },
  "tpot_s": 0.0371,
  "batch": {
    "1": 34.7,
    "4": 52.6,
    "8": 104.9
  }
}


In [14]:
from google.colab import files

files.download("baselines.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
%run verify_cell.py

ttft lengths: ['128', '2048', '512'], tpot_s: 0.0371
batch tokens/s 1/4/8: 34.7/52.6/104.9
KV measured 28.0 KB/token vs formula 28.0 KB/token
GREEN CHECK: PASS




---



---



---



In [16]:
def kv_kb_per_token(
    layers=28,
    kv_heads=2,
    head_dim=128,
    dbytes=2
):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024


KB_PER_TOKEN = kv_kb_per_token()

BLOCK_TOKENS = 16

BLOCK_KB = BLOCK_TOKENS * KB_PER_TOKEN

print(
    "KB per token:", KB_PER_TOKEN,
    " KB per block:", BLOCK_KB
)

KB per token: 28.0  KB per block: 448.0


In [17]:
class SlabAllocator:

    def __init__(self, budget_kb, max_len=4096):
        self.budget_kb = budget_kb
        self.max_len = max_len

        self.used_kb = 0

        # seq_id -> reserved_kb
        self.resident = {}


    def admit(self, seq_id):

        need = self.max_len * KB_PER_TOKEN

        if self.used_kb + need > self.budget_kb:
            return False

        self.used_kb += need
        self.resident[seq_id] = need

        return True


    def complete(self, seq_id):

        self.used_kb -= self.resident.pop(seq_id)

In [18]:
class BlockPoolAllocator:

    def __init__(self, budget_kb, block_kb=BLOCK_KB):

        self.block_kb = block_kb

        self.total_blocks = int(
            budget_kb // block_kb
        )

        self.free_blocks = self.total_blocks

        # seq_id -> number of blocks held
        self.block_tables = {}


    def admit(self, seq_id):

        # Need at least one block to start
        if self.free_blocks < 1:
            return False

        self.free_blocks -= 1
        self.block_tables[seq_id] = 1

        return True


    def grow(self, seq_id, current_len_tokens):

        # Ceiling division
        needed_blocks = -(
            -current_len_tokens // BLOCK_TOKENS
        )

        held = self.block_tables[seq_id]

        if needed_blocks > held:

            extra = needed_blocks - held

            if self.free_blocks < extra:
                return False

            self.free_blocks -= extra
            self.block_tables[seq_id] = needed_blocks

        return True


    def complete(self, seq_id):

        self.free_blocks += self.block_tables.pop(seq_id)

In [19]:
import random

random.seed(7)


def make_workload(n_sequences=60, max_len=4096):

    lengths = []

    for _ in range(n_sequences):

        if random.random() < 0.85:

            # Most requests are short
            lengths.append(
                random.randint(50, 400)
            )

        else:

            # Occasional long straggler
            lengths.append(
                random.randint(2000, max_len)
            )

    return lengths


WORKLOAD = make_workload()

print(
    "mean length:",
    sum(WORKLOAD) / len(WORKLOAD),
    "max:",
    max(WORKLOAD)
)

mean length: 444.4 max: 3763


In [20]:
def simulate_slab(budget_kb, workload):
    alloc = SlabAllocator(budget_kb)

    admitted, rejected = 0, 0

    for i, length in enumerate(workload):
        if alloc.admit(seq_id=i):
            admitted += 1
        else:
            rejected += 1

    return {
        "peak_concurrent": admitted,
        "admitted": admitted,
        "rejected": rejected
    }


def simulate_blockpool(budget_kb, workload):
    alloc = BlockPoolAllocator(budget_kb)

    admitted, rejected = 0, 0

    for i, length in enumerate(workload):

        if not alloc.admit(seq_id=i):
            rejected += 1
            continue

        grew = True

        for step_len in range(
            BLOCK_TOKENS,
            length + BLOCK_TOKENS,
            BLOCK_TOKENS
        ):
            if not alloc.grow(
                seq_id=i,
                current_len_tokens=min(step_len, length)
            ):
                grew = False
                break

        if grew:
            admitted += 1
        else:
            alloc.complete(seq_id=i)
            rejected += 1

    return {
        "peak_concurrent": admitted,
        "admitted": admitted,
        "rejected": rejected
    }


BUDGET_KB = 2 * 1024 * 1024

slab_result = simulate_slab(
    BUDGET_KB,
    WORKLOAD
)

blockpool_result = simulate_blockpool(
    BUDGET_KB,
    WORKLOAD
)

print("slab:      ", slab_result)
print("block-pool:", blockpool_result)

slab:       {'peak_concurrent': 18, 'admitted': 18, 'rejected': 42}
block-pool: {'peak_concurrent': 60, 'admitted': 60, 'rejected': 0}


In [21]:
import json

report = {
    "kb_per_token": KB_PER_TOKEN,
    "block_tokens": BLOCK_TOKENS,
    "budget_kb": BUDGET_KB,
    "workload_mean_len": sum(WORKLOAD) / len(WORKLOAD),
    "slab": slab_result,
    "blockpool": blockpool_result,
    "blockpool_advantage": round(
        blockpool_result["peak_concurrent"] /
        slab_result["peak_concurrent"],
        2
    ),
}

with open("kv_sim_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

{
  "kb_per_token": 28.0,
  "block_tokens": 16,
  "budget_kb": 2097152,
  "workload_mean_len": 444.4,
  "slab": {
    "peak_concurrent": 18,
    "admitted": 18,
    "rejected": 42
  },
  "blockpool": {
    "peak_concurrent": 60,
    "admitted": 60,
    "rejected": 0
  },
  "blockpool_advantage": 3.33
}


In [22]:
from google.colab import files

files.download("kv_sim_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>